# 7.2 模型怎麼知道輪到誰？


要讓助手接著回答「1+1=?」，輸入應停在哪裡？不能把還不知道的「2」先放進去，也不能只停在問題末尾。先用 EOS 結束 user 段，再放 assistant 標記，讓下一步正好預測回答的第一個 token。

本字表 BOS=1、EOS=2、user=3、assistant=4，普通內容的 byte 加 8。生成之前先送進模型的這段前文叫 **prompt**。下面直接拼出它，再和 [7.1](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html) 訓練時的輸入 X 比較。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import ByteTokenizer, render_chat

tok = ByteTokenizer()
question = "1+1=?"
prompt = [tok.bos_id, tok.user_id] + tok.encode(question) + [tok.eos_id, tok.assistant_id]
print(prompt)
print("最後一項", prompt[-1])
x, _ = render_chat([{"role": "user", "content": question}, {"role": "assistant", "content": "2"}])
print("訓練輸入X的前九項", x[:9].tolist())
assert prompt[-1] == tok.assistant_id
assert prompt == x[:9].tolist()

清單的 `+` 表示按順序相接，不是逐項相加，所以 prompt 是 `[1,3,57,51,57,69,71,2,4]`，共九個 token。`prompt[-1]` 取最後一項 4，第一個 assert 確認它是 assistant 標記。這裡沒有放入答案 2，因為生成時還不知道它；模型要從最後一個位置（assistant）的 logits 預測回答的第一個 token。`x, _ = ...` 的 `_` 表示 Y 這次用不到。印出的 X 前九項和 prompt 完全相同，第二個 assert 核對這件事：訓練時，X 在 assistant 之後才接上答案；推論時就停在 assistant，讓模型接著寫。

為什麼 user 段的 EOS 和 assistant 標記都要保留？兩者功能不同：EOS 宣告前一段話結束，assistant 指定下一段由誰說。訓練資料的每一筆都是「…EOS、assistant、回答」的順序，模型學到的是在這個結構之後開始回答。若推論時換了邊界順序，例如省略 EOS 或改用別的角色標記，模型遇到的是沒有練過的結構，不能因為字面看得懂就期待它接對。

多輪對話的歷史可以包含完整的舊回答，但當前這一輪只放到 assistant 為止。這份結構只告訴模型輪到誰，不提供算術能力；回答內容對不對，仍要用新題檢查。

練習只刪掉 prompt 那一行尾端的 `tok.assistant_id`。先預測：最後一項變成 EOS（2），模型的當前位置停在問題結尾，而不是我們指定的回答角色；第一個 assert 就會失敗並停下。再執行核對。這不是要修掉的程式錯誤，而是檢查發現 prompt 不符合約定。恢復後兩個檢查都通過。
